# NHS A&E Data Preparation — From Raw Excel to Analysis-Ready Dataset

**Project:** AI-Driven Operational Decision Support in NHS Emergency Care

This notebook converts the raw NHS England A&E workbook into a **clean, one-row-per-month dataset** for:

**Data Cleaning → EDA → Forecasting → Automation → Power BI Dashboard**

> This notebook focuses on **data preparation**.

In [1]:
# Install the Excel reader

%pip install -q pandas xlrd

Note: you may need to restart the kernel to use updated packages.


In [2]:
# import necessary python libraries

import pandas as pd
import numpy as np

# print version for python installed library
print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)

pandas version: 2.2.2
numpy version: 1.26.4


In [ ]:
# Read the sheets which existing in the workbook

RAW_FILE = "Dataset/01_raw_Monthly-AE-Time-Series.xls"

excel_file = pd.ExcelFile(RAW_FILE)

print("Available sheets:")
print(excel_file.sheet_names)

Available sheets:
['Activity', 'Performance', 'Booking', 'Notes', 'Chart Data', 'Charts']


In [4]:
# Read the Activity sheet from the whole workbook

activity_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Activity",
    header=13
)

print("Rows and columns:", activity_raw.shape)
display(activity_raw.head())

Rows and columns: (191, 18)


,Unnamed: 0,Period,Type 1 Departments - Major A&E,Type 2 Departments - Single Specialty,Type 3 Departments - Other A&E/Minor Injury Unit,Total Attendances,Emergency Admissions via Type 1 A&E,Emergency Admissions via Type 2 A&E,Emergency Admissions via Type 3 and 4 A&E,Total Emergency Admissions via A&E,Other Emergency Admissions (i.e not via A&E),Total Emergency Admissions,Number of patients spending >4 hours from decision to admit to admission,Number of patients spending >12 hours from decision to admit to admission,Unnamed: 14,Operational standard (Performance),Unnamed: 16,0.95
0,NaN,2010-08-01,1.138652e+06,54371.000000,559358.000000,1.752381e+06,287438.000000,5367.000000,8081.000000,300886.000000,124816.000000,425702.000000,3697.000000,1.0,NaN,0.95,NaN,NaN
1,NaN,2010-09-01,1.150728e+06,55181.000000,550359.000000,1.756268e+06,293991.000000,5543.000000,3673.000000,303207.000000,121693.000000,424900.000000,5907.000000,0.0,NaN,0.95,NaN,NaN
2,NaN,2010-10-01,1.163143e+06,54961.000000,583244.000000,1.801348e+06,303452.000000,5485.000000,2560.000000,311497.000000,124718.000000,436215.000000,6932.000000,0.0,NaN,0.95,NaN,NaN
3,NaN,2010-11-01,1.111295e+06,53727.428571,486005.428571,1.651027e+06,297832.000000,5731.142857,3279.000000,306842.142857,122256.857143,429099.000000,7179.000000,2.0,NaN,0.95,NaN,NaN
4,NaN,2010-12-01,1.159204e+06,45536.428571,533000.857143,1.737741e+06,318602.428571,6277.000000,3198.428571,328077.857143,124650.857143,452728.714286,13818.142857,15.0,NaN,0.95,NaN,NaN


In [5]:
# Remove the blank first column - clean activity sheet

activity = activity_raw.iloc[:, 1:].copy()

display(activity.head())

,Period,Type 1 Departments - Major A&E,Type 2 Departments - Single Specialty,Type 3 Departments - Other A&E/Minor Injury Unit,Total Attendances,Emergency Admissions via Type 1 A&E,Emergency Admissions via Type 2 A&E,Emergency Admissions via Type 3 and 4 A&E,Total Emergency Admissions via A&E,Other Emergency Admissions (i.e not via A&E),Total Emergency Admissions,Number of patients spending >4 hours from decision to admit to admission,Number of patients spending >12 hours from decision to admit to admission,Unnamed: 14,Operational standard (Performance),Unnamed: 16,0.95
0,2010-08-01,1.138652e+06,54371.000000,559358.000000,1.752381e+06,287438.000000,5367.000000,8081.000000,300886.000000,124816.000000,425702.000000,3697.000000,1.0,NaN,0.95,NaN,NaN
1,2010-09-01,1.150728e+06,55181.000000,550359.000000,1.756268e+06,293991.000000,5543.000000,3673.000000,303207.000000,121693.000000,424900.000000,5907.000000,0.0,NaN,0.95,NaN,NaN
2,2010-10-01,1.163143e+06,54961.000000,583244.000000,1.801348e+06,303452.000000,5485.000000,2560.000000,311497.000000,124718.000000,436215.000000,6932.000000,0.0,NaN,0.95,NaN,NaN
3,2010-11-01,1.111295e+06,53727.428571,486005.428571,1.651027e+06,297832.000000,5731.142857,3279.000000,306842.142857,122256.857143,429099.000000,7179.000000,2.0,NaN,0.95,NaN,NaN
4,2010-12-01,1.159204e+06,45536.428571,533000.857143,1.737741e+06,318602.428571,6277.000000,3198.428571,328077.857143,124650.857143,452728.714286,13818.142857,15.0,NaN,0.95,NaN,NaN


In [6]:
# Rename Activity columns

activity.columns = [
    "Period",
    "MajorAE_Attendances",
    "SingleSpecialty_Attendances",
    "MinorInjuryUnit_Attendances",
    "TotalAttendances",
    "Type1_Admissions",
    "Type2_Admissions",
    "Type3_4_Admissions",
    "TotalAdmissions_viaAE",
    "OtherAdmissions",
    "TotalAdmissions",
    "Over4hr_DecisionToAdmit",
    "Over12hr_DecisionToAdmit",
    "blank1",
    "OpStandard",
    "blank2",
    "blank3",
]

print(activity.columns.tolist())

['Period', 'MajorAE_Attendances', 'SingleSpecialty_Attendances', 'MinorInjuryUnit_Attendances', 'TotalAttendances', 'Type1_Admissions', 'Type2_Admissions', 'Type3_4_Admissions', 'TotalAdmissions_viaAE', 'OtherAdmissions', 'TotalAdmissions', 'Over4hr_DecisionToAdmit', 'Over12hr_DecisionToAdmit', 'blank1', 'OpStandard', 'blank2', 'blank3']


In [7]:
# Remove rows without a date and create a real datetime column

activity = activity[activity["Period"].notna()].copy()

activity["Month"] = pd.to_datetime(
    activity["Period"],
    errors="coerce"
)

print("Missing Month values:", activity["Month"].isna().sum())
display(activity[["Period", "Month"]].head())

Missing Month values: 0


,Period,Month
0,2010-08-01,2010-08-01
1,2010-09-01,2010-09-01
2,2010-10-01,2010-10-01
3,2010-11-01,2010-11-01
4,2010-12-01,2010-12-01


In [8]:
# Read and clean the Performance sheet

performance_raw = pd.read_excel(
    RAW_FILE,
    sheet_name="Performance",
    header=13
)

performance = performance_raw.iloc[:, 1:].copy()

performance.columns = [
    "Period",
    "Type1_u4",
    "Type2_u4",
    "Type3_u4",
    "Total_u4",
    "Type1_o4",
    "Type2_o4",
    "Type3_o4",
    "Total_o4",
    "PercentWithin4hrs",
    "Pct_Type1",
    "Pct_Type2",
    "Pct_Type3",
]

performance = performance[performance["Period"].notna()].copy()

performance["Month"] = pd.to_datetime(
    performance["Period"],
    errors="coerce"
)

display(performance.head())

,Period,Type1_u4,Type2_u4,Type3_u4,Total_u4,Type1_o4,Type2_o4,Type3_o4,Total_o4,PercentWithin4hrs,Pct_Type1,Pct_Type2,Pct_Type3,Month
0,2010-11-01,1.065456e+06,53584.000000,485550.857143,1.604591e+06,45838.428571,143.428571,454.571429,46436.428571,0.971874,0.958752,0.997330,0.999065,2010-11-01
1,2010-12-01,1.070729e+06,45395.857143,531699.428571,1.647824e+06,88475.285714,140.571429,1301.428571,89917.285714,0.948256,0.923676,0.996913,0.997558,2010-12-01
2,2011-01-01,1.061898e+06,51420.571429,541588.857143,1.654907e+06,71982.571429,164.285714,742.428571,72889.285714,0.957814,0.936517,0.996815,0.998631,2011-01-01
3,2011-02-01,1.007385e+06,51153.428571,493883.000000,1.552421e+06,46322.428571,95.857143,524.571429,46942.857143,0.970649,0.956039,0.998130,0.998939,2011-02-01
4,2011-03-01,1.167090e+06,57694.857143,579483.571429,1.804268e+06,58132.142857,205.571429,835.000000,59172.714286,0.968245,0.952554,0.996450,0.998561,2011-03-01


In [9]:
# Perform left join on Activity and Performance sheets with Month column 

df = activity.merge(
    performance[["Month", "PercentWithin4hrs", "Pct_Type1", "Pct_Type2", "Pct_Type3"]],
    on="Month",
    how="left"
)

# Keep only the columns this project needs
df = df[
    [
        "Month", "TotalAttendances", "MajorAE_Attendances",
        "SingleSpecialty_Attendances", "MinorInjuryUnit_Attendances",
        "TotalAdmissions", "Over12hr_DecisionToAdmit",
        "PercentWithin4hrs", "Pct_Type1", "Pct_Type2", "Pct_Type3",
    ]
].sort_values("Month").reset_index(drop=True)

# relabels columns for readability
df = df.rename(columns={
    "Over12hr_DecisionToAdmit": "Breach12hr",
    "Pct_Type1": "MajorInjuryPerformancePct",
    "Pct_Type2": "SingleSpecialityPerformancePct",
    "Pct_Type3": "MinorInjuryPerformancePct",
})

print("Shape:", df.shape)
df.head()

Shape: (191, 11)


,Month,TotalAttendances,MajorAE_Attendances,SingleSpecialty_Attendances,MinorInjuryUnit_Attendances,TotalAdmissions,Breach12hr,PercentWithin4hrs,MajorInjuryPerformancePct,SingleSpecialityPerformancePct,MinorInjuryPerformancePct
0,2010-08-01,1.752381e+06,1.138652e+06,54371.000000,559358.000000,425702.000000,1.0,NaN,NaN,NaN,NaN
1,2010-09-01,1.756268e+06,1.150728e+06,55181.000000,550359.000000,424900.000000,0.0,NaN,NaN,NaN,NaN
2,2010-10-01,1.801348e+06,1.163143e+06,54961.000000,583244.000000,436215.000000,0.0,NaN,NaN,NaN,NaN
3,2010-11-01,1.651027e+06,1.111295e+06,53727.428571,486005.428571,429099.000000,2.0,0.971874,0.958752,0.997330,0.999065
4,2010-12-01,1.737741e+06,1.159204e+06,45536.428571,533000.857143,452728.714286,15.0,0.948256,0.923676,0.996913,0.997558


In [10]:
# Admission Rate: what share of everyone who attends A&E ends up admitted
# to hospital. This is about case *severity*, separate from volume.
df["AdmissionRate"] = (df["TotalAdmissions"] / df["TotalAttendances"] * 100).round(2)

# 12-hour breach rate: one of the most closely watched NHS statistics
# nationally right now — patients waiting far beyond the 4-hour target.
df["Breach12hrRate"] = (df["Breach12hr"] / df["TotalAttendances"] * 100).round(3)

# The Type 1/2/3 performance columns arrived as 0–1 fractions (e.g. 0.603) —
# convert to the same 0–100 percentage scale as everything else.
df["PercentWithin4hrs"] = (df["PercentWithin4hrs"] * 100).round(1)
df["MajorInjuryPerformancePct"] = (df["MajorInjuryPerformancePct"] * 100).round(1)
df["SingleSpecialityPerformancePct"] = (df["SingleSpecialityPerformancePct"] * 100).round(1)
df["MinorInjuryPerformancePct"] = (df["MinorInjuryPerformancePct"] * 100).round(1)

df[["Month", "AdmissionRate", "Breach12hrRate", "PercentWithin4hrs",
    "MajorInjuryPerformancePct", "SingleSpecialityPerformancePct", "MinorInjuryPerformancePct"]].tail(6)

,Month,AdmissionRate,Breach12hrRate,PercentWithin4hrs,MajorInjuryPerformancePct,SingleSpecialityPerformancePct,MinorInjuryPerformancePct
185,2026-01-01,23.55,3.084,72.5,57.3,95.1,97.1
186,2026-02-01,23.28,2.581,74.1,59.4,96.0,97.4
187,2026-03-01,22.77,1.919,77.1,64.1,96.3,97.3
188,2026-04-01,22.41,2.036,76.9,63.8,95.6,97.5
189,2026-05-01,21.70,2.043,75.7,61.9,96.0,97.2
190,2026-06-01,21.93,2.029,75.0,61.2,96.2,96.7


In [11]:
# Category shares based on different types of injury
df["MajorAE_SharePct"] = (df["MajorAE_Attendances"] / df["TotalAttendances"] * 100).round(2)
df["SingleSpecialty_SharePct"] = (df["SingleSpecialty_Attendances"] / df["TotalAttendances"] * 100).round(2)
df["MinorInjuryUnit_SharePct"] = (df["MinorInjuryUnit_Attendances"] / df["TotalAttendances"] * 100).round(2)

df[["Month", "MajorAE_SharePct", "SingleSpecialty_SharePct", "MinorInjuryUnit_SharePct"]].head()

,Month,MajorAE_SharePct,SingleSpecialty_SharePct,MinorInjuryUnit_SharePct
0,2010-08-01,64.98,3.10,31.92
1,2010-09-01,65.52,3.14,31.34
2,2010-10-01,64.57,3.05,32.38
3,2010-11-01,67.31,3.25,29.44
4,2010-12-01,66.71,2.62,30.67


In [12]:
# Data-quality checks

print("Missing values per column:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("Duplicate months:", df["Month"].duplicated().sum())
print("\nDate range:", df["Month"].min(), "to", df["Month"].max())
print("Number of rows:", len(df))

Missing values per column:
Month                             0
TotalAttendances                  0
MajorAE_Attendances               0
SingleSpecialty_Attendances       0
MinorInjuryUnit_Attendances       0
TotalAdmissions                   0
Breach12hr                        0
PercentWithin4hrs                 3
MajorInjuryPerformancePct         3
SingleSpecialityPerformancePct    3
MinorInjuryPerformancePct         3
AdmissionRate                     0
Breach12hrRate                    0
MajorAE_SharePct                  0
SingleSpecialty_SharePct          0
MinorInjuryUnit_SharePct          0
dtype: int64

Duplicate rows: 0
Duplicate months: 0

Date range: 2010-08-01 00:00:00 to 2026-06-01 00:00:00
Number of rows: 191


In [13]:
# Add month number and season column for seasonal behaviour

season_map = {
    12: "Winter", 1: "Winter", 2: "Winter",
    3: "Spring", 4: "Spring", 5: "Spring",
    6: "Summer", 7: "Summer", 8: "Summer",
    9: "Autumn", 10: "Autumn", 11: "Autumn"
}

df["MonthNum"] = df["Month"].dt.month
df["Season"] = df["MonthNum"].map(season_map)

display(df.head(15))

,Month,TotalAttendances,MajorAE_Attendances,SingleSpecialty_Attendances,MinorInjuryUnit_Attendances,TotalAdmissions,Breach12hr,PercentWithin4hrs,MajorInjuryPerformancePct,SingleSpecialityPerformancePct,MinorInjuryPerformancePct,AdmissionRate,Breach12hrRate,MajorAE_SharePct,SingleSpecialty_SharePct,MinorInjuryUnit_SharePct,MonthNum,Season
0,2010-08-01,1.752381e+06,1.138652e+06,54371.000000,559358.000000,425702.000000,1.000000,NaN,NaN,NaN,NaN,24.29,0.000,64.98,3.10,31.92,8,Summer
1,2010-09-01,1.756268e+06,1.150728e+06,55181.000000,550359.000000,424900.000000,0.000000,NaN,NaN,NaN,NaN,24.19,0.000,65.52,3.14,31.34,9,Autumn
2,2010-10-01,1.801348e+06,1.163143e+06,54961.000000,583244.000000,436215.000000,0.000000,NaN,NaN,NaN,NaN,24.22,0.000,64.57,3.05,32.38,10,Autumn
3,2010-11-01,1.651027e+06,1.111295e+06,53727.428571,486005.428571,429099.000000,2.000000,97.2,95.9,99.7,99.9,25.99,0.000,67.31,3.25,29.44,11,Autumn
4,2010-12-01,1.737741e+06,1.159204e+06,45536.428571,533000.857143,452728.714286,15.000000,94.8,92.4,99.7,99.8,26.05,0.001,66.71,2.62,30.67,12,Winter
5,2011-01-01,1.727797e+06,1.133881e+06,51584.857143,542331.285714,442003.714286,17.285714,95.8,93.7,99.7,99.9,25.58,0.001,65.63,2.99,31.39,1,Winter
6,2011-02-01,1.599364e+06,1.053707e+06,51249.285714,494407.571429,401206.428571,2.714286,97.1,95.6,99.8,99.9,25.09,0.000,65.88,3.20,30.91,2,Winter
7,2011-03-01,1.863441e+06,1.225222e+06,57900.428571,580318.571429,446845.571429,0.571429,96.8,95.3,99.6,99.9,23.98,0.000,65.75,3.11,31.14,3,Spring
8,2011-04-01,1.844375e+06,1.197213e+06,54042.428571,593119.714286,419243.285714,6.428571,96.9,95.3,99.5,99.9,22.73,0.000,64.91,2.93,32.16,4,Spring
9,2011-05-01,1.873695e+06,1.221687e+06,57067.000000,594940.714286,427276.571429,2.571429,97.1,95.6,99.3,99.9,22.80,0.000,65.20,3.05,31.75,5,Spring


In [14]:
# Calculate the recent 3-year seasonal baseline

def seasonal_baseline_3yr(row, data):
    window = data[
        (data["MonthNum"] == row["MonthNum"]) &
        (data["Month"] < row["Month"]) &
        (data["Month"] >= row["Month"] - pd.DateOffset(years=3))
    ]

    if len(window) == 0:
        return np.nan

    return window["TotalAttendances"].mean()

df["SeasonalBaselineAvg"] = df.apply(
    lambda row: seasonal_baseline_3yr(row, df),
    axis=1
)

display(
    df[["Month", "TotalAttendances", "Season", "SeasonalBaselineAvg"]].head(20)
)

,Month,TotalAttendances,Season,SeasonalBaselineAvg
0,2010-08-01,1.752381e+06,Summer,NaN
1,2010-09-01,1.756268e+06,Autumn,NaN
2,2010-10-01,1.801348e+06,Autumn,NaN
3,2010-11-01,1.651027e+06,Autumn,NaN
4,2010-12-01,1.737741e+06,Winter,NaN
5,2011-01-01,1.727797e+06,Winter,NaN
6,2011-02-01,1.599364e+06,Winter,NaN
7,2011-03-01,1.863441e+06,Spring,NaN
8,2011-04-01,1.844375e+06,Spring,NaN
9,2011-05-01,1.873695e+06,Spring,NaN


In [15]:
# Create the risk flag based on average seasonalbaseline

def risk_flag(row):
    if pd.isna(row["SeasonalBaselineAvg"]):
        return "N/A - not enough prior years yet"

    ratio = row["TotalAttendances"] / row["SeasonalBaselineAvg"]

    if ratio > 1.10:
        return "High"
    elif ratio > 1.03:
        return "Medium"

    return "Low"

df["RiskFlag"] = df.apply(risk_flag, axis=1)

display(
    df[
        [
            "Month",
            "TotalAttendances",
            "SeasonalBaselineAvg",
            "RiskFlag"
        ]
    ].tail(12)
)

,Month,TotalAttendances,SeasonalBaselineAvg,RiskFlag
179,2025-07-01,2408866.0,2.235425e+06,Medium
180,2025-08-01,2266937.0,2.088777e+06,Medium
181,2025-09-01,2310222.0,2.130917e+06,Medium
182,2025-10-01,2401266.0,2.257628e+06,Medium
183,2025-11-01,2347046.0,2.220462e+06,Medium
184,2025-12-01,2326139.0,2.274977e+06,Low
185,2026-01-01,2319296.0,2.139098e+06,Medium
186,2026-02-01,2117450.0,2.053391e+06,Medium
187,2026-03-01,2432272.0,2.308412e+06,Medium
188,2026-04-01,2345329.0,2.190710e+06,Medium


In [16]:
# Feature Engineering for the forecasting model
df["MonthSin"] = np.sin(2 * np.pi * df["MonthNum"] / 12)
df["MonthCos"] = np.cos(2 * np.pi * df["MonthNum"] / 12)

df["MajorShare_x_Winter"] = df["MajorAE_SharePct"] * (df["Season"] == "Winter").astype(int)

df["Attendances_Lag1"] = df["TotalAttendances"].shift(1)
df["Attendances_RollingMean3"] = df["TotalAttendances"].rolling(3).mean()

df[["Month", "MonthSin", "MonthCos", "MajorShare_x_Winter",
    "Attendances_Lag1", "Attendances_RollingMean3"]].tail(8)

,Month,MonthSin,MonthCos,MajorShare_x_Winter,Attendances_Lag1,Attendances_RollingMean3
183,2025-11-01,-5.000000e-01,8.660254e-01,0.00,2401266.0,2.352845e+06
184,2025-12-01,-2.449294e-16,1.000000e+00,61.83,2347046.0,2.358150e+06
185,2026-01-01,5.000000e-01,8.660254e-01,61.86,2326139.0,2.330827e+06
186,2026-02-01,8.660254e-01,5.000000e-01,61.30,2319296.0,2.254295e+06
187,2026-03-01,1.000000e+00,6.123234e-17,0.00,2117450.0,2.289673e+06
188,2026-04-01,8.660254e-01,-5.000000e-01,0.00,2432272.0,2.298350e+06
189,2026-05-01,5.000000e-01,-8.660254e-01,0.00,2345329.0,2.411666e+06
190,2026-06-01,1.224647e-16,-1.000000e+00,0.00,2457398.0,2.413544e+06


In [17]:
# Final clean dataset

final_df = (
    df.drop(columns=["MonthNum"])
      .sort_values("Month")
      .reset_index(drop=True)
)

# Add Year column from the Month date
final_df.insert(1, "Year", final_df["Month"].dt.year)

print("Final shape:", final_df.shape)
print("Columns:", final_df.columns.tolist())
final_df.tail(10)

Final shape: (191, 25)
Columns: ['Month', 'Year', 'TotalAttendances', 'MajorAE_Attendances', 'SingleSpecialty_Attendances', 'MinorInjuryUnit_Attendances', 'TotalAdmissions', 'Breach12hr', 'PercentWithin4hrs', 'MajorInjuryPerformancePct', 'SingleSpecialityPerformancePct', 'MinorInjuryPerformancePct', 'AdmissionRate', 'Breach12hrRate', 'MajorAE_SharePct', 'SingleSpecialty_SharePct', 'MinorInjuryUnit_SharePct', 'Season', 'SeasonalBaselineAvg', 'RiskFlag', 'MonthSin', 'MonthCos', 'MajorShare_x_Winter', 'Attendances_Lag1', 'Attendances_RollingMean3']


,Month,Year,TotalAttendances,MajorAE_Attendances,SingleSpecialty_Attendances,MinorInjuryUnit_Attendances,TotalAdmissions,Breach12hr,PercentWithin4hrs,MajorInjuryPerformancePct,...,SingleSpecialty_SharePct,MinorInjuryUnit_SharePct,Season,SeasonalBaselineAvg,RiskFlag,MonthSin,MonthCos,MajorShare_x_Winter,Attendances_Lag1,Attendances_RollingMean3
181,2025-09-01,2025,2310222.0,1417440.0,50733.0,842049.0,535580.0,44765.0,75.1,61.1,...,2.20,36.45,Autumn,2.130917e+06,Medium,-1.000000e+00,-1.836970e-16,0.00,2266937.0,2.328675e+06
182,2025-10-01,2025,2401266.0,1481222.0,54756.0,865288.0,553499.0,54326.0,74.2,60.1,...,2.28,36.03,Autumn,2.257628e+06,Medium,-8.660254e-01,5.000000e-01,0.00,2310222.0,2.326142e+06
183,2025-11-01,2025,2347046.0,1450144.0,49725.0,847177.0,531747.0,50648.0,74.2,60.3,...,2.12,36.10,Autumn,2.220462e+06,Medium,-5.000000e-01,8.660254e-01,0.00,2401266.0,2.352845e+06
184,2025-12-01,2025,2326139.0,1438207.0,47522.0,840410.0,542188.0,50789.0,73.8,59.6,...,2.04,36.13,Winter,2.274977e+06,Low,-2.449294e-16,1.000000e+00,61.83,2347046.0,2.358150e+06
185,2026-01-01,2026,2319296.0,1434757.0,50710.0,833829.0,546136.0,71517.0,72.5,57.3,...,2.19,35.95,Winter,2.139098e+06,Medium,5.000000e-01,8.660254e-01,61.86,2326139.0,2.330827e+06
186,2026-02-01,2026,2117450.0,1298005.0,48233.0,771212.0,493015.0,54649.0,74.1,59.4,...,2.28,36.42,Winter,2.053391e+06,Medium,8.660254e-01,5.000000e-01,61.30,2319296.0,2.254295e+06
187,2026-03-01,2026,2432272.0,1478144.0,55393.0,898735.0,553892.0,46665.0,77.1,64.1,...,2.28,36.95,Spring,2.308412e+06,Medium,1.000000e+00,6.123234e-17,0.00,2117450.0,2.289673e+06
188,2026-04-01,2026,2345329.0,1430825.0,52714.0,861790.0,525660.0,47750.0,76.9,63.8,...,2.25,36.74,Spring,2.190710e+06,Medium,8.660254e-01,-5.000000e-01,0.00,2432272.0,2.298350e+06
189,2026-05-01,2026,2457398.0,1493345.0,52820.0,911233.0,533280.0,50212.0,75.7,61.9,...,2.15,37.08,Spring,2.353247e+06,Medium,5.000000e-01,-8.660254e-01,0.00,2345329.0,2.411666e+06
190,2026-06-01,2026,2437906.0,1488602.0,54890.0,894414.0,534753.0,49466.0,75.0,61.2,...,2.25,36.69,Summer,2.290457e+06,Medium,1.224647e-16,-1.000000e+00,0.00,2457398.0,2.413544e+06


In [18]:
# Save the clean Dataset

OUTPUT_FILE = "Dataset/02_clean_Monthly_AE_data.csv"

final_df.to_csv(
    OUTPUT_FILE,
    index=False
)

print(f"Saved {len(final_df)} rows and {len(final_df.columns)} columns to {OUTPUT_FILE}")

Saved 191 rows and 25 columns to Dataset/02_clean_Monthly_AE_data.csv
